![alt text](./Cerny_logo_1.jpg)

# Analysis of Cerny ventilation recordings

## Further processing of HFOV data for cases `AT000001-AT001251`

#### Author: Dr Gusztav Belteki
##### email: gbelteki@aol.com

### 1. Import the necessary libraries and set options

In [1]:
import IPython
import pandas as pd
import numpy as np
import scipy as sp
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.dates as dates

import os
import sys
import pickle

from pandas import Series, DataFrame
from datetime import datetime, timedelta
from scipy import stats

%matplotlib inline
matplotlib.style.use('classic')
matplotlib.rcParams['figure.facecolor'] = 'w'

pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 300)
pd.set_option('mode.chained_assignment', None) 

In [2]:
print("Python version: {}".format(sys.version))
print("pandas version: {}".format(pd.__version__))
print("matplotlib version: {}".format(matplotlib.__version__))
print("NumPy version: {}".format(np.__version__))
print("IPython version: {}".format(IPython.__version__))

Python version: 3.12.11 | packaged by Anaconda, Inc. | (main, Jun  5 2025, 08:03:38) [Clang 14.0.6 ]
pandas version: 2.2.2
matplotlib version: 3.9.2
NumPy version: 1.26.4
IPython version: 9.11.0


In [3]:
# Define styling of boxplots
medianprops = {'color':'black', 'linewidth':2}; boxprops = {'color':'black', 'linestyle':'-'}
whiskerprops = {'color':'black', 'linestyle':'-'}; capprops = {'color':'black', 'linestyle':'-'}
flierprops = {'color':'black', 'marker': '.'}
meanprops =  {'marker':'D', 'markeredgecolor':'black', 'markerfacecolor':'black'}

### 2. List and set the working directory and the directory to write out data

In [4]:
# Topic of the Notebook which will also be the name of the subfolder containing results
TOPIC = 'HFOV_2026'

# Path to project folder containing clinical data (current weights only) and for export of results
PATH = os.path.join(os.sep, 'Users', 'guszti', 'Library', 'Mobile Documents', 'com~apple~CloudDocs', 
                            'Documents', 'Research', 'Ventilation')

# Name of the external hard drive
DRIVE = 'GB_1'

# Folder containing the file with the manually collected clinical data
DIR_READ_CLIN = os.path.join(os.sep, PATH, 'ventilation_fabian_new')

# Processed data loaded from external drive
DIR_READ = os.path.join(os.sep, 'Volumes', DRIVE, 'data_dump', 'fabian_new')

# Results will be written in this folder
DIR_WRITE =  os.path.join(os.sep, PATH, 'ventilation_fabian_new', 'Analyses', TOPIC)
os.makedirs(DIR_WRITE, exist_ok=True)

# Images and raw data will be written on an external hard drive
DATA_DUMP = os.path.join(os.sep, 'Volumes', DRIVE, 'data_dump', 'fabian_new', TOPIC)
os.makedirs(DATA_DUMP, exist_ok=True)

In [5]:
DIR_READ_CLIN, DIR_READ, DIR_WRITE, DATA_DUMP,

('/Users/guszti/Library/Mobile Documents/com~apple~CloudDocs/Documents/Research/Ventilation/ventilation_fabian_new',
 '/Volumes/GB_1/data_dump/fabian_new',
 '/Users/guszti/Library/Mobile Documents/com~apple~CloudDocs/Documents/Research/Ventilation/ventilation_fabian_new/Analyses/HFOV_2026',
 '/Volumes/GB_1/data_dump/fabian_new/HFOV_2026')

### 3. Import processed ventilator data

These recordings all have >10 minutes of HFOV data

In [6]:
# These data only include HFOV
with open(os.path.join(DATA_DUMP, 'data_pars_measurements_hfov_only.pickle'), 'rb') as handle:
    parameters = pickle.load(handle)
with open(os.path.join(DATA_DUMP, 'data_pars_settings_hfov_only.pickle'), 'rb') as handle:
    settings = pickle.load(handle)
with open(os.path.join(DATA_DUMP, 'data_pars_alarms_hfov_only.pickle'), 'rb') as handle:
    alarms = pickle.load(handle)

In [7]:
len(parameters), len(settings), len(alarms)

(138, 138, 138)

### 4. Combine ventilator parameters and settings

In [8]:
hfov_data = {}
for case in parameters:
    hfov_data[case] = pd.concat([parameters[case], settings[case]], axis=1)

In [9]:
hfov_data_all = pd.concat(hfov_data)
hfov_data_all.index.set_names(['case', 'datetime'], inplace=True)

In [10]:
hfov_data_all.head()

PIP   MAP     MV  Leak  VTimand  deltaP  \
case     datetime                                                        
AT000033 2020-11-20 09:24:31  34.5  16.0  1.795  34.0      3.9    33.9   
         2020-11-20 09:24:33  33.2  15.1  1.823  34.0      3.8    32.4   
         2020-11-20 09:24:35  32.4  14.7  1.884  30.0      3.7    30.9   
         2020-11-20 09:24:37  32.2  14.7  1.986   0.0      3.6    29.9   
         2020-11-20 09:24:39  32.0  14.8  2.025   0.0      3.6    29.1   

                              VThf_emand   DCO2  HFOV_freq  FiO2  RR_CO2  \
case     datetime                                                          
AT000033 2020-11-20 09:24:31         3.8  144.0       10.0  97.1   255.0   
         2020-11-20 09:24:33         3.8  144.0       10.0  97.1   255.0   
         2020-11-20 09:24:35         3.8  144.0       10.0  97.1   255.0   
         2020-11-20 09:24:37         3.7  136.0       10.0  97.1   255.0   
         2020-11-20 09:24:39         3.6  129.0       10.0  97.1   255.0   

                                 MV_kg  VTimand_kg  VThf_emand_kg   DCO2_kg2  \
case     datetime                                                              
AT000033 2020-11-20 09:24:31  1.196667    2.600000       2.533333  64.000000   
         2020-11-20 09:24:33  1.215333    2.533333       2.533333  64.000000   
         2020-11-20 09:24:35  1.256000    2.466667       2.533333  64.000000   
         2020-11-20 09:24:37  1.324000    2.400000       2.466667  60.444444   
         2020-11-20 09:24:39  1.350000    2.400000       2.400000  57.333333   

                              deltaP_diff  HFOV_freq_diff  FiO2_diff  \
case     datetime                                                      
AT000033 2020-11-20 09:24:31         -1.1             0.0       -2.9   
         2020-11-20 09:24:33         -2.6             0.0       -2.9   
         2020-11-20 09:24:35         -4.1             0.0       -2.9   
         2020-11-20 09:24:37         -5.1             0.0       -2.9   
         2020-11-20 09:24:39         -5.9             0.0       -2.9   

                              MAP_diff VThf_diff VThf_diff_kg Patient_range  \
case     datetime                                                             
AT000033 2020-11-20 09:24:31       1.0      12.4         13.6      Neonatal   
         2020-11-20 09:24:33       0.1      11.5         12.7      Neonatal   
         2020-11-20 09:24:35      -0.3      11.1         12.3      Neonatal   
         2020-11-20 09:24:37      -0.3      11.1         12.3      Neonatal   
         2020-11-20 09:24:39      -0.2      11.2         12.4      Neonatal   

                             Ventilator_mode  PIP_set  FiO2_set  \
case     datetime                                                 
AT000033 2020-11-20 09:24:31             HFO     15.0     100.0   
         2020-11-20 09:24:33             HFO     15.0     100.0   
         2020-11-20 09:24:35             HFO     15.0     100.0   
         2020-11-20 09:24:37             HFO     15.0     100.0   
         2020-11-20 09:24:39             HFO     15.0     100.0   

                              Flow_insp_set  deltaP_set  Freq_set_HFOV  \
case     datetime                                                        
AT000033 2020-11-20 09:24:31           10.0        35.0           10.0   
         2020-11-20 09:24:33           10.0        35.0           10.0   
         2020-11-20 09:24:35           10.0        35.0           10.0   
         2020-11-20 09:24:37           10.0        35.0           10.0   
         2020-11-20 09:24:39           10.0        35.0           10.0   

                              MAP_set_HFOV Volume_limit_set VG_set Powerstate  \
case     datetime                                                               
AT000033 2020-11-20 09:24:31          15.0              off    3.6    Battery   
         2020-11-20 09:24:33          15.0              off    3.6    Battery   
         2020-11-20 09:24:35          15.0    

### 5. Only keep parameters relevant for HFOV

In [17]:
print(sorted(hfov_data_all.columns))

['Apnea_time_set', 'Battery_rem_pc', 'Battery_rem_time', 'Bias_flow', 'C20_C', 'CO2_baropressure_set', 'Cdyn', 'Cdyn_kg', 'DCO2', 'DCO2_kg2', 'DCO2_lim_high_set', 'DCO2_lim_high_set_kg2', 'DCO2_lim_low_set', 'DCO2_lim_low_set_kg2', 'FOT_running', 'FiO2', 'FiO2_diff', 'FiO2_flush_set', 'FiO2_flush_time_set', 'FiO2_set', 'Flow_demand', 'Flow_exp', 'Flow_exp_set', 'Flow_insp', 'Flow_insp_set', 'Flow_sensor_state', 'Freq_set_HFOV', 'HFOV_freq', 'HFOV_freq_diff', 'HFO_flow', 'HFO_recruitment', 'IE_E_set', 'IE_I_set', 'I_E_HFOV', 'Leak', 'Leak_comp', 'Leakage_lim_set', 'MAP', 'MAP_diff', 'MAP_set_HFOV', 'MV', 'MV_kg', 'MV_lim_high_set', 'MV_lim_high_set_kg', 'MV_lim_low_set', 'MV_lim_low_set_kg', 'MVresp', 'Measuring_unit_CO2_set', 'Measuring_unit_pressure_set', 'Oxy_sensor_state', 'PEEP', 'PEEP_lim_low_set', 'PEEP_set', 'PIP', 'PIP_lim_high_set', 'PIP_set', 'PIP_set_PSV', 'P_man_breath_CPAP_HFO_set', 'P_man_breath_duoPAP_NCPAP_set', 'Patient_range', 'Powerstate', 'Pressure_rise_control', 'R

In [18]:
cols_to_keep = ['DCO2', 'DCO2_kg2',  'FOT_running', 'FiO2', 'FiO2_diff',  'FiO2_set', 'Flow_exp', 'Flow_exp_set', 
                'Flow_insp', 'Flow_insp_set', 'Freq_set_HFOV', 'HFOV_freq', 'HFOV_freq_diff', 'HFO_flow', 'HFO_recruitment', 
                'I_E_HFOV', 'Leak', 'Leak_comp', 'MAP', 'MAP_set_HFOV', 'P_man_breath_CPAP_HFO_set', 'P_man_breath_duoPAP_NCPAP_set',
                'VG_state',  'VThf_diff', 'VThf_diff_kg', 'VThf_emand', 'VThf_emand_kg', 'VTimand', 'VTimand_kg', 'Ventilation_stopped', 
                'Ventilator_mode',  'deltaP', 'deltaP_diff', 'deltaP_set',]

In [19]:
hfov_data_all = hfov_data_all[cols_to_keep]

### 6. Export processed ventilator data

In [21]:
%%time

# Only recordings selected as containing >10% of both HFOV-VG and HFOV-noVG, and only HFOV parts
with open(os.path.join(DATA_DUMP, 'hfov_data_all.pickle'), 'wb') as handle:
    pickle.dump(hfov_data_all, handle, protocol=pickle.HIGHEST_PROTOCOL)

CPU times: user 47.5 ms, sys: 39.4 ms, total: 86.8 ms
Wall time: 172 ms
